# Entwicklung der Ausschank-Betriebe in Bayern

Nachbau eines gestapelten Balkendiagramms (Destatis / Umsatzsteuerstatistik, **schematische Annäherung** an die im Originalbild ablesbaren Summen — keine offizielle Tabellenabfrage).

Gestaltung: hoher **Data-Ink**-Anteil (kein Rahmen oben/rechts, dezentes horizontales Raster, klare Typografie).

## Referenz (Original)

![](bayern_ausschank_reference.png)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

%matplotlib inline

plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans", "Helvetica Neue", "Arial"],
        "font.size": 10,
        "axes.linewidth": 0.55,
    }
)

# --- Approximate totals (read from published chart; rounded) ---
years = np.arange(2014, 2025)
totals = np.array(
    [
        5480,
        5460,
        5475,
        5455,
        5490,
        5620,
        3880,
        3380,
        4120,
        4320,
        4520,
    ],
    dtype=float,
)

# Stable category shares (sum = 1); scaled to each year's total
labels_de = [
    "Vergnügungslokale",
    "Diskotheken und Tanzlokale",
    "Bars",
    "Sonstige getränkegeprägte Gastronomie",
    "Schankwirtschaften",
]
base = np.array([118.0, 355.0, 445.0, 865.0, 3717.0])
base /= base.sum()
parts = totals[:, np.newaxis] * base[np.newaxis, :]

df = pd.DataFrame(parts, columns=labels_de, index=years.astype(int))
df["Summe"] = df.sum(axis=1)

colors = {
    "Vergnügungslokale": "#6d7d42",
    "Diskotheken und Tanzlokale": "#d0e238",
    "Bars": "#c9d8f0",
    "Sonstige getränkegeprägte Gastronomie": "#2f62c4",
    "Schankwirtschaften": "#13264d",
}

fig, ax = plt.subplots(figsize=(10, 5.8), dpi=140)
ax.set_facecolor("white")

bottom = np.zeros(len(years))
x = years.astype(float)
width = 0.72

for col in labels_de:
    h = df[col].values
    ax.bar(
        x,
        h,
        width,
        bottom=bottom,
        label=col,
        color=colors[col],
        edgecolor="white",
        linewidth=0.35,
    )
    bottom += h

ax.set_xlim(2013.35, 2024.65)
ax.set_xticks(np.arange(2014, 2025, 2))
ax.set_xticklabels([str(int(y)) for y in range(2014, 2025, 2)])
ax.set_ylabel("Anzahl Betriebe")

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda v, _: f"{int(v):,}".replace(",", "."))
)
ax.set_yticks(np.arange(1000, 6001, 1000))
ax.set_ylim(0, 6200)

ax.yaxis.grid(True, linestyle="-", linewidth=0.4, color="0.78", alpha=0.9)
ax.set_axisbelow(True)
for s in ax.spines.values():
    s.set_visible(False)
ax.axhline(0, color="0.35", linewidth=0.6, clip_on=False)
ax.tick_params(axis="both", length=3, width=0.55, colors="0.25")

ax.text(
    0.0,
    1.14,
    "Entwicklung der Ausschank-Betriebe in Bayern",
    transform=ax.transAxes,
    fontsize=13,
    fontweight="bold",
    color="0.1",
    va="bottom",
)
ax.text(
    0.0,
    1.06,
    "Anzahl der umsatzsteuerpflichtigen Unternehmen (Voranmeldung) im Gastgewerbe nach Kategorien.",
    transform=ax.transAxes,
    fontsize=9.5,
    color="0.25",
    va="bottom",
)

# Legend: first row at top = bottom-of-stack category (Matplotlib default order matches plot order)
handles, leg_labels = ax.get_legend_handles_labels()
ax.legend(
    handles,
    leg_labels,
    loc="upper left",
    bbox_to_anchor=(0.01, 0.99),
    bbox_transform=ax.transAxes,
    frameon=False,
    fontsize=8.5,
    labelspacing=0.35,
)

cut = 2019.5
ax.axvline(cut, color="0.55", linestyle=(0, (4, 3)), linewidth=1.0, ymax=0.98, zorder=5)
ax.text(
    cut + 0.12,
    5350,
    "Einbruch durch\nCorona-Pandemie",
    fontsize=9,
    color="0.35",
    ha="left",
    va="center",
)

fig.text(
    0.09,
    0.02,
    "Quelle: Statistisches Bundesamt | Umsatzsteuerstatistik (Werte im Diagramm näherungsweise dem Original angeglichen)",
    fontsize=8,
    color="0.4",
)

plt.tight_layout(rect=[0, 0.06, 1, 0.82])
plt.show()

df.round(0).astype(int)

## Kurznotiz

- Für **exakte** Zahlen bitte die offizielle Tabelle des Statistischen Bundesamtes verwenden; hier sind die **Balkenhöhen** nur an die veröffentlichte Grafik angeglichen.
- Die **gestrichelte Linie** zwischen 2019 und 2020 markiert den pandemiebedingten Einbruch wie im Original.